In [1]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

Current Project Root Directory set to: /Users/luca/Documents/GitHub/US_conflict_forecasting
Current Search Path set to: ['/Users/luca/Documents/GitHub/US_conflict_forecasting', '/Users/luca/Documents/GitHub/US_conflict_forecasting/1_Features/b_Conflict_Features/Conflict_Actors', '/Users/luca/anaconda3/envs/ML_Lab/lib/python312.zip', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/lib-dynload', '', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages', '/Users/luca/Documents/GitHub/US_conflict_forecasting']


In [11]:
import pandas as pd

#Loading data
ACLED_df = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/joined_ACLED_ZIP_22_PA.csv'))

In [12]:
#Function to parse and clean actor names
def actor_names(actor_string):

    #checking if input is string
    if isinstance(actor_string, str):

        #splitting string by semicolon + deleting whitespace to extract actor names 
        actors = [actor.strip() for actor in actor_string.split(';')]

        #splitting further by colon if possible to keep main name of actor 
        actors = [actor.split(':')[-1].strip() for actor in actors]
        return actors
    
    else: 

        #return empty list 
        return[]

In [13]:
#applying function + expanding list into separate rows so each actors becomes individual data point 
actor_list_expanded = ACLED_df['assoc_actor_1'].apply(actor_names).explode().reset_index(drop=True)

#grouping by actor names and counting occurence
actor_counts = actor_list_expanded.value_counts()


print(actor_counts)

assoc_actor_1
Labor Group (United States)                218
Government of the United States (2021-)    172
Democratic Party                           164
Students (United States)                   139
Women (United States)                      130
                                          ... 
Workers World Party                          1
IfNotNow                                     1
Private Security Forces (United States)      1
Everytown for Gun Safety                     1
Abolish ICE                                  1
Name: count, Length: 150, dtype: int64


In [14]:
#converting series to df + naming cols
actor_counts_df = actor_counts.reset_index()
actor_counts_df.columns = ['Actor', 'Count']

#converting to csv (currently not saved as csv)
#actor_counts_df.to_csv("actor_counts.csv", index=False)

#### Creating columns for presence of actor types 

In [15]:
#Loading data
actor_actorytype_df = pd.read_excel(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/ACLED_Actor_ActorType_mapping.xlsx'))

In [17]:
#Joining count with mapping (actually just nice to have, counts not necessary for features)
#merged_actor_mapping = pd.merge(left=actor_counts_df, right=actor_actorytype, on=['Actor'], how='inner')

In [16]:
#Creating dictionary for mapping actor names to types
actor_type_map = actor_actorytype_df.set_index('Actor')['actor_type'].to_dict()

In [17]:
#Applying the function and get a list of types per row
ACLED_df['Actor_Types'] = ACLED_df['assoc_actor_1'].apply(actor_names).apply(lambda actors: [actor_type_map.get(actor) for actor in actors if actor in actor_type_map])

In [18]:
#Creating binary columns for each actor type (defined in actor_actorytype_df)
for type_id in sorted(actor_actorytype_df['actor_type'].unique()):
    ACLED_df[f'actor_type{type_id}'] = ACLED_df['Actor_Types'].apply(lambda types: 1 if type_id in types else 0)

In [19]:
#Dropping all columns except the relevant Actor features and ID
columns_to_keep = ['event_id_cnty', 'ZCTA5CE10'] + [f'actor_type{i}' for i in range(1, 10)]
ACLED_df_sparse = ACLED_df[columns_to_keep]

In [21]:
#Saving as csv
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "ACLED_ZIP_Actor_22.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

ACLED_df_sparse.to_csv(output_path_1, index=False)